# 1. Shape Compare approach

In [85]:
# VISUAL MOTOR INTEGRATION (IMAGE BASED)

import cv2
import numpy as np
# !pip install scikit-image

In [86]:
# LOAD IDEAL SHAPES (TEMPLATES)
def load_templates():
    templates = {
        "vertical_line": cv2.imread("templates/vertical_line.png", 0),
        "horizontal_line": cv2.imread("templates/horizontal_line.png", 0),
        "circle": cv2.imread("templates/circle.png", 0),
        "cross": cv2.imread("templates/cross.png", 0),
        "square": cv2.imread("templates/square.png", 0),
        "x": cv2.imread("templates/x.png", 0),
        "triangle": cv2.imread("templates/triangle.png", 0),
        "diamond": cv2.imread("templates/diamond.png", 0)
    }
    # print(len(templates))
    return templates

In [87]:
# SPLIT INTO 2x4 GRID
def split_grid(img):

    h, w = img.shape[:2]

    rows = 2
    cols = 4

    cell_h = h // rows
    cell_w = w // cols

    cells = []

    for i in range(rows):
        for j in range(cols):
            y1 = i * cell_h
            y2 = (i+1) * cell_h
            x1 = j * cell_w
            x2 = (j+1) * cell_w

            cell = img[y1:y2, x1:x2]
            cells.append(cell)

    return cells

In [88]:
# PREPROCESS (stable)
def preprocess(img):
    if len(img.shape) == 3:
        img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    img = cv2.GaussianBlur(img, (5,5), 0)

    _, th = cv2.threshold(img, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)

    # ensure white shape on black background
    white_pixels = np.sum(th == 255)
    black_pixels = np.sum(th == 0)

    if white_pixels > black_pixels:
        th = cv2.bitwise_not(th)

    # remove tiny noise
    kernel = np.ones((3,3), np.uint8)
    th = cv2.morphologyEx(th, cv2.MORPH_OPEN, kernel)

    return th

In [89]:
# EXTRACT SHAPE (crop)
def extract(th):
    cnts, _ = cv2.findContours(th, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not cnts:
        return None

    cnt = max(cnts, key=cv2.contourArea)
    x,y,w,h = cv2.boundingRect(cnt)
    return th[y:y+h, x:x+w]


In [90]:
# NORMALIZE (important)
def normalize(img):

    canvas = np.zeros((128,128), dtype=np.uint8)

    h, w = img.shape
    scale = 100 / max(h, w)

    new_w = int(w * scale)
    new_h = int(h * scale)

    resized = cv2.resize(img, (new_w, new_h))

    x_offset = (128 - new_w)//2
    y_offset = (128 - new_h)//2

    canvas[y_offset:y_offset+new_h, x_offset:x_offset+new_w] = resized

    return canvas

In [91]:
def line_type(shape):

    pts = np.column_stack(np.where(shape > 0))

    if len(pts) < 10:
        return "unknown"

    y_min, x_min = pts.min(axis=0)
    y_max, x_max = pts.max(axis=0)

    h = y_max - y_min + 1
    w = x_max - x_min + 1

    # aspect ratio
    if h > 4 * w:
        return "vertical_line"

    if w > 4 * h:
        return "horizontal_line"

    return "other"

In [92]:
import cv2
import numpy as np

def shape_similarity(img1, img2):

    th1 = preprocess(img1)
    th2 = preprocess(img2)

    crop1 = extract(th1)
    crop2 = extract(th2)

    if crop1 is None or crop2 is None:
        return 0.0

    norm1 = normalize(crop1)
    norm2 = normalize(crop2)

    # STRICT LINE GATE
    type1 = line_type(norm1)
    type2 = line_type(norm2)

    # if one is vertical/horizontal and other is not
    if type1 != type2:
        if "line" in type1 or "line" in type2:
            return 0.01

    # EDGES (shape only)
    edge1 = cv2.Canny(norm1, 50, 150)
    edge2 = cv2.Canny(norm2, 50, 150)

    # DISTANCE TRANSFORM (bidirectional)
    dist1 = cv2.distanceTransform(255 - edge1, cv2.DIST_L2, 3)
    dist2 = cv2.distanceTransform(255 - edge2, cv2.DIST_L2, 3)

    pts1 = np.column_stack(np.where(edge1 > 0))
    pts2 = np.column_stack(np.where(edge2 > 0))

    if len(pts1) == 0 or len(pts2) == 0:
        return 0.0

    # A → B
    d1 = dist2[pts1[:,0], pts1[:,1]].mean()

    # B → A
    d2 = dist1[pts2[:,0], pts2[:,1]].mean()

    dist = (d1 + d2) / 2

    # FINAL SCORE (tunable)
    score = np.exp(-dist / 2.5)

    return float(score)

In [93]:
# QUALITY METRICS
def compute_quality(shape):

    edges = cv2.Canny(shape, 50, 150)

    contours, _ = cv2.findContours(edges, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    # smoothness
    perimeters = [cv2.arcLength(c, True) for c in contours]
    smoothness = np.std(perimeters) if perimeters else 1

    # closure
    closed = sum([1 for c in contours if cv2.isContourConvex(c)])

    # proportion
    h, w = shape.shape
    ratio = w / (h + 1e-6)

    return smoothness, closed, ratio

In [94]:
# QUALITY SCORE
# -------------------------------
def quality_score_fn(metrics):

    smoothness, closure, ratio = metrics

    score = 0

    # smoothness
    if smoothness < 10: score += 2
    elif smoothness < 20: score += 1

    # closure
    if closure >= 1: score += 1

    # proportion
    if 0.7 < ratio < 1.3: score += 2
    elif 0.5 < ratio < 1.5: score += 1

    return max(1, min(5, score))

In [95]:
# ACCURACY SCORE
def acc_score_cal(correct, age_group):

    acc_score = 0
    if age_group == "6-7":
        if correct >= 7: acc_score = 5
        elif correct >= 5: acc_score = 4
        elif correct >= 3: acc_score = 3
        elif correct >= 2: acc_score = 2
        else: acc_score = 1
    else:
        if correct == 8: acc_score = 5
        elif correct >= 6: acc_score = 4
        elif correct >= 4: acc_score = 3
        elif correct >= 2: acc_score = 2
        else: acc_score = 1
    return acc_score

In [105]:
# MAIN FUNCTION
def vmi_grid_test(image_path, age_group="6-7"):

    img = cv2.imread(image_path)

    # split grid
    cells = split_grid(img)

    templates = load_templates()

    correct = 0
    quality_scores = []
    template_order = ["vertical_line","horizontal_line","circle","cross","square","x","triangle","diamond"]

    for i in range(8):

        cell = cells[i]

        thresh = preprocess(cell)

        key = template_order[i]
        # -------- FETCH TEMPLATE --------
        if key not in templates:
            print(f"❌ Missing template: {key}")
            quality_scores.append(1)
            continue

        template = templates[key]

        if template is None or not isinstance(template, np.ndarray):
            print(f"❌ Invalid template: {key}")
            quality_scores.append(1)
            continue

        # accuracy
        # sim = match_shape(thresh, template)
        sim = shape_similarity(thresh, template)
        print(f"Shape is {key} and Score is {sim:0.2f}")

        if sim > 0.5:
            correct += 1

        # quality
        metrics = compute_quality(thresh)
        q = quality_score_fn(metrics)
        quality_scores.append(q)

        # debug view
        cv2.imshow(f"Accurate Shape {i+1}", template)
        cv2.waitKey(1000)
        cv2.imshow(f"Shape {i+1}", thresh)
        cv2.waitKey(1000)
        # cv2.destroyAllWindows()
    cv2.destroyAllWindows()

    # ACCURACY SCORE
    acc_score = acc_score_cal(correct, age_group)

    # QUALITY SCORE
    quality_score = round(np.mean(quality_scores)) if quality_scores else 1

    # FINAL SCORE
    final_score = round((acc_score + quality_score) / 2)

    print("------ VMI GRID RESULT ------")
    print(f"Shapes Correct: {correct}/8")
    print(f"Accuracy Score: {acc_score}")
    print(f"Quality Score: {quality_score}")
    print(f"Final Score: {final_score}")

    return final_score

In [110]:
image_path = "data/raw_img.png"
print(vmi_grid_test(image_path, age_group="6-7"))

Shape is vertical_line and Score is 0.01
Shape is horizontal_line and Score is 0.01
Shape is circle and Score is 0.00
Shape is cross and Score is 0.01
Shape is square and Score is 0.05
Shape is x and Score is 0.00
Shape is triangle and Score is 0.01
Shape is diamond and Score is 0.01
------ VMI GRID RESULT ------
Shapes Correct: 0/8
Accuracy Score: 1
Quality Score: 3
Final Score: 2
2


In [111]:
# template_order = ["vertical_line","horizontal_line","circle","cross","square","x","triangle","diamond"]

# for i in range(8):
#     path = "templates/" + template_order[i] + ".png"
#     img1 = cv2.imread(path)
#     for j in range(i, 8):
#         path = "templates/" + template_order[j] + ".png"
#         img2 = cv2.imread(path)
#         score = shape_similarity(img1, img2)
#         print(f"{template_order[i]}-{template_order[j]}: Score: {score:0.3}")
#         # bar = "█" * int(score * 40)
#         # print(f"  {shape:18s}: {score:.4f}  {bar}")

In [112]:
# img1 = cv2.imread("templates/square.png")
# img2 = cv2.imread("templates/square.png")

# score = shape_similarity(img1, img2)

# print("Similarity:", score)